# Fixation mRNN 100-Initialization Current Consistency

Train or load 100 independent initializations of the 50-unit-per-region region-PC mRNN with temporal derivative weight 5 and curvature weight 3. The notebook submits missing fits as a dSQ array on the `gpu` partition, then summarizes whether the reconstruction and current-geometry analyses are stable across random seeds.

## 1. Resource Estimate

For this model size, one array task trains one seed. The model has about 50K trainable parameters and a small full-batch target, so memory pressure is low; runtime is the main constraint.

In [ ]:
import pandas as pd

recommended_resources = pd.DataFrame(
    [
        {
            "resource": "per array task",
            "gpu": "1",
            "cpus_per_task": 2,
            "mem_per_cpu": "8G",
            "total_ram": "16G",
            "time_limit": "02:00:00",
            "notes": "Good starting request for 100K iterations on the gpu partition.",
        },
        {
            "resource": "lean request",
            "gpu": "1",
            "cpus_per_task": 1,
            "mem_per_cpu": "8G",
            "total_ram": "8G",
            "time_limit": "02:00:00",
            "notes": "Likely enough RAM, but leaves less CPU headroom for imports and checkpoint IO.",
        },
    ]
)
recommended_resources

## 2. Setup

In [ ]:
from pathlib import Path
import json
import shlex

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

repo_root = Path.cwd()
if not (repo_root / "src").exists():
    repo_root = next(parent for parent in Path.cwd().parents if (parent / "src").exists())

import sys
if str(repo_root / "src") not in sys.path:
    sys.path.insert(0, str(repo_root / "src"))

from dal_monte_2022_analysis.ephys.modeling import (
    backproject_replay_outputs_to_firing_rates,
    extract_fixation_latent_dynamics,
    extract_region_current_vectors,
    load_fixation_mrnn_config,
    make_targets,
    pc_reconstructed_firing_rate_accuracy,
    reconstruction_accuracy,
    replay_fixation_mrnn_run,
    resolve_fixation_mrnn_output_root,
    settings_from_config,
)
from dal_monte_2022_analysis.ephys.modeling.fixation_mrnn_training import load_or_create_seed_plan
from dal_monte_2022_analysis.runtime.hpc.jobs import (
    submit_dsq_array_job,
    track_job_completion,
    write_job_file,
)

plt.rcParams.update({"figure.dpi": 140, "axes.spines.top": False, "axes.spines.right": False})

## 3. Ensemble Settings

In [ ]:
cfg = load_fixation_mrnn_config(repo_root / "configs/ephys_fixation_mrnn.yaml")

settings = settings_from_config(
    cfg,
    overrides={
        "target_mode": "region_pcs",
        "temporal_basis_count": 0,
        "hidden_units": 50,
        "lr": 1e-3,
        "epochs": 100_000,
        "initialization_mode": "single",
        "device": "cuda",
        "temporal_derivative_loss_scale": 5.0,
        "temporal_curvature_loss_scale": 3.0,
        "correlation_loss_scale": 0.0,
        "variance_loss_scale": 0.0,
        "fr_reconstruction_loss_scale": 0.0,
        "fr_temporal_derivative_loss_scale": 0.0,
        "fr_temporal_curvature_loss_scale": 0.0,
        "l1_weight_scale": 0.0,
        "l1_rate_scale": 0.0,
        "l2_weight_scale": 0.0,
        "l2_rate_scale": 0.0,
        "gradient_clip_norm": 1.0,
        "divergence_loss_threshold": 1e6,
        "divergence_patience": 100,
        "divergence_min_iteration": 100,
    },
)

n_initializations = 100
max_divergence_retries = 5
scratch_id = "ensemble_d5_c3_50u_100k_100init"
rerun_scratch_id_suggestion = "ensemble_d5_c3_50u_100k_100init_clip1_div1e6"
output_root = resolve_fixation_mrnn_output_root(settings) / "scratch"
ensemble_dir = output_root / scratch_id
job_dir = repo_root / "hpc" / "fixation_mrnn_ensemble_d5_c3"
log_dir = job_dir / "logs"
job_file_path = job_dir / "joblist.txt"
sbatch_script_path = job_dir / "submit.sbatch"

targets = make_targets(settings)
timeline = np.asarray(targets.timeline_s, dtype=float)

seed_plan = load_or_create_seed_plan(
    ensemble_dir,
    settings,
    n_seeds=n_initializations,
    overwrite=False,
)
run_table = pd.DataFrame(
    [
        {
            "init_idx": idx,
            "seed": int(seed),
            "run_dir": str(ensemble_dir / f"init={idx:03d}_seed={int(seed)}"),
            "checkpoint_path": str(ensemble_dir / f"init={idx:03d}_seed={int(seed)}" / "checkpoint_final.pth"),
            "history_path": str(ensemble_dir / f"init={idx:03d}_seed={int(seed)}" / "history.csv"),
        }
        for idx, seed in enumerate(seed_plan)
    ]
)
run_table["is_complete"] = run_table["checkpoint_path"].map(lambda p: Path(p).exists())
run_table["history_exists"] = run_table["history_path"].map(lambda p: Path(p).exists())
display(run_table["is_complete"].value_counts(dropna=False).rename("n_runs"))
run_table.head()

## 4. Submit Missing Array Jobs

Rerunning this cell will not resubmit completed checkpoints. It writes commands only for missing runs, then submits those commands as one dSQ array job.

In [ ]:
submit_missing_jobs = True
track_after_submit = True
partition = "gpu"
gres = "gpu:1"
cpus_per_task = 2
mem_per_cpu = "16G"
time_limit = "02:00:00"


def build_training_command(row):
    scratch_child = f"{scratch_id}/init={int(row.init_idx):03d}_seed={int(row.seed)}"
    argv = [
        "python",
        "-u",
        "scripts/ephys/modeling/train_fixation_mrnn.py",
        "--mrnn-cfg",
        "configs/ephys_fixation_mrnn.yaml",
        "--scratch-id",
        scratch_child,
        "--target-mode",
        "region_pcs",
        "--epochs",
        str(settings.epochs),
        "--lr",
        str(settings.lr),
        "--seed",
        str(int(row.seed)),
        "--initialization-mode",
        "single",
        "--device",
        "cuda",
        "--hidden-units",
        str(settings.hidden_units),
        "--temporal-basis-count",
        str(settings.temporal_basis_count),
        "--temporal-derivative-loss-scale",
        str(settings.temporal_derivative_loss_scale),
        "--temporal-curvature-loss-scale",
        str(settings.temporal_curvature_loss_scale),
        "--correlation-loss-scale",
        "0",
        "--variance-loss-scale",
        "0",
        "--fr-reconstruction-loss-scale",
        "0",
        "--fr-temporal-derivative-loss-scale",
        "0",
        "--fr-temporal-curvature-loss-scale",
        "0",
        "--l1-weight-scale",
        "0",
        "--l1-rate-scale",
        "0",
        "--l2-weight-scale",
        "0",
        "--l2-rate-scale",
        "0",
        "--gradient-clip-norm",
        str(settings.gradient_clip_norm),
        "--divergence-loss-threshold",
        str(settings.divergence_loss_threshold),
        "--divergence-patience",
        str(settings.divergence_patience),
        "--divergence-min-iteration",
        str(settings.divergence_min_iteration),
        "--max-divergence-retries",
        str(max_divergence_retries),
    ]
    segments = [
        "module load miniconda",
        "module load CUDA/12.1.1",
        "conda deactivate || true",
        "conda activate gaze_processing",
        f"cd {shlex.quote(str(repo_root))}",
        shlex.join(argv),
    ]
    return " && ".join(segments)

missing = run_table.loc[~run_table["is_complete"]].copy()
commands = [build_training_command(row) for row in missing.itertuples(index=False)]

job_id = None
if commands:
    write_job_file(job_file_path, commands)
    print(f"Missing runs: {len(commands)}")
    if submit_missing_jobs:
        job_id = submit_dsq_array_job(
            job_file_path=job_file_path,
            sbatch_script_path=sbatch_script_path,
            log_dir=log_dir,
            job_name="fix_mrnn_d5_c3_100init",
            partition=partition,
            cpus_per_task=cpus_per_task,
            mem_per_cpu=mem_per_cpu,
            time_limit=time_limit,
            gres=gres,
        )
        if track_after_submit:
            track_job_completion(job_id, poll_secs=60, log_every_secs=300)
else:
    print("All 100 checkpoints are present; no jobs submitted.")

job_id

## 5. Reload Completed Runs

After the array finishes, rerun this cell and the analysis cells below. The notebook requires all 100 histories/checkpoints for ensemble SEM plots.

In [ ]:
def read_run_seed(run_dir, fallback_seed):
    manifest_path = Path(run_dir) / "manifest.json"
    if manifest_path.exists():
        with manifest_path.open("r", encoding="utf-8") as f:
            manifest = json.load(f)
        if manifest.get("status", "complete") != "failed" and "seed" in manifest:
            return int(manifest["seed"])
    return int(fallback_seed)


run_table["is_complete"] = run_table["checkpoint_path"].map(lambda p: Path(p).exists())
run_table["history_exists"] = run_table["history_path"].map(lambda p: Path(p).exists())
complete = run_table[run_table["is_complete"] & run_table["history_exists"]].copy()
print(f"Complete runs: {len(complete)} / {n_initializations}")
if len(complete) != n_initializations:
    display(run_table.loc[~(run_table["is_complete"] & run_table["history_exists"]), ["init_idx", "seed", "run_dir"]].head(20))
    raise RuntimeError("Not all ensemble runs are complete yet. Re-run the submit cell or wait for the array job to finish.")

history_frames = []
for row in complete.itertuples(index=False):
    hist = pd.read_csv(row.history_path)
    hist["init_idx"] = int(row.init_idx)
    hist["seed"] = read_run_seed(row.run_dir, row.seed)
    hist["planned_seed"] = int(row.seed)
    history_frames.append(hist)
all_history = pd.concat(history_frames, ignore_index=True)
final_history = all_history.sort_values("iteration").groupby("init_idx", as_index=False).tail(1)
best_row = final_history.loc[final_history["loss"].idxmin()]
best_init_idx = int(best_row["init_idx"])
best_run_dir = Path(complete.loc[complete["init_idx"] == best_init_idx, "run_dir"].iloc[0])
print(f"Best init: {best_init_idx:03d}; final loss={best_row['loss']:.6g}; run_dir={best_run_dir}")
final_history.sort_values("loss").head(10)

## 6. Training Health: Best and Worst Seeds

This is the first sanity check for the completed ensemble. The mean loss trajectory is only interpretable if most seeds train into the same basin. Runs are flagged when losses become non-finite, exceed a blow-up threshold, fail to improve, or finish much worse than the best decile.

In [ ]:
blowup_loss_threshold = 1e6
stuck_improvement_fraction = 0.05
best_decile_final = float(final_history["loss"].quantile(0.10))
stuck_final_loss_multiplier = 10.0

health_rows = []
for init_idx, hist in all_history.groupby("init_idx"):
    hist = hist.sort_values("iteration")
    losses = hist["loss"].to_numpy(dtype=float)
    finite = np.isfinite(losses)
    initial_loss = float(losses[0]) if len(losses) else np.nan
    final_loss = float(losses[-1]) if len(losses) else np.nan
    min_loss = float(np.nanmin(losses)) if len(losses) else np.nan
    max_loss = float(np.nanmax(losses)) if len(losses) else np.nan
    improvement_fraction = (initial_loss - final_loss) / max(abs(initial_loss), 1e-12) if np.isfinite(initial_loss) and np.isfinite(final_loss) else np.nan
    run_seed = int(hist["seed"].iloc[0]) if "seed" in hist else -1
    diverged = (not finite.all()) or (np.isfinite(max_loss) and max_loss > blowup_loss_threshold)
    failed_to_improve = np.isfinite(improvement_fraction) and improvement_fraction < stuck_improvement_fraction
    bad_final = np.isfinite(final_loss) and final_loss > stuck_final_loss_multiplier * best_decile_final
    health_rows.append(
        {
            "init_idx": int(init_idx),
            "seed": run_seed,
            "initial_loss": initial_loss,
            "final_loss": final_loss,
            "min_loss": min_loss,
            "max_loss": max_loss,
            "improvement_fraction": improvement_fraction,
            "nonfinite_points": int((~finite).sum()),
            "diverged_or_blew_up": bool(diverged),
            "failed_to_improve": bool(failed_to_improve),
            "bad_final_loss": bool(bad_final),
            "likely_bad_run": bool(diverged or failed_to_improve or bad_final),
        }
    )

training_health = pd.DataFrame(health_rows).sort_values("final_loss")
display(training_health["likely_bad_run"].value_counts(dropna=False).rename("n_runs"))
display(training_health.head(10))
display(training_health.tail(10))

best5 = training_health.nsmallest(5, "final_loss")["init_idx"].tolist()
worst5 = training_health.nlargest(5, "final_loss")["init_idx"].tolist()

fig, axes = plt.subplots(1, 2, figsize=(12.0, 3.8), sharex=True, sharey=True)
for ax, init_indices, title in [(axes[0], best5, "5 best final losses"), (axes[1], worst5, "5 worst final losses")]:
    for init_idx in init_indices:
        hist = all_history[all_history["init_idx"] == init_idx].sort_values("iteration")
        seed = int(hist["seed"].iloc[0])
        ax.plot(hist["iteration"], hist["loss"], linewidth=1.0, label=f"init {init_idx:03d}, seed {seed}")
    ax.axhline(blowup_loss_threshold, color="crimson", linestyle="--", linewidth=0.9, label="blow-up threshold")
    ax.set(title=title, xlabel="iteration", ylabel="loss")
    ax.set_yscale("log")
    ax.legend(frameon=False, fontsize=6)
fig.tight_layout()

## 7. Best and Ensemble Loss Trajectories

In [ ]:
loss_columns = [
    "loss",
    "reconstruction_loss",
    "temporal_derivative_loss",
    "temporal_curvature_loss",
]
plot_history = all_history.copy()
max_points = 1200
stride = max(1, int(np.ceil(plot_history["iteration"].nunique() / max_points)))
plot_history = plot_history[plot_history["iteration"] % stride == 0]

summary = (
    plot_history.groupby("iteration")[loss_columns]
    .agg(["mean", "sem"])
    .reset_index()
)
summary.columns = ["iteration"] + [f"{col}_{stat}" for col, stat in summary.columns[1:]]
best_history = plot_history[plot_history["init_idx"] == best_init_idx].sort_values("iteration")

fig, axes = plt.subplots(1, 2, figsize=(12, 3.5), sharex=True)
for column in loss_columns:
    if column not in best_history:
        continue
    axes[0].plot(best_history["iteration"], best_history[column], linewidth=1.0, label=column)
axes[0].set(title=f"Best fit init {best_init_idx:03d}", xlabel="iteration", ylabel="loss")
axes[0].set_yscale("log")
axes[0].legend(frameon=False, fontsize=7)

for column in loss_columns:
    mean = summary[f"{column}_mean"].to_numpy(dtype=float)
    sem = summary[f"{column}_sem"].to_numpy(dtype=float)
    x = summary["iteration"].to_numpy(dtype=float)
    axes[1].plot(x, mean, linewidth=1.0, label=column)
    axes[1].fill_between(x, mean - sem, mean + sem, alpha=0.14)
axes[1].set(title="100-initialization mean +/- SEM", xlabel="iteration", ylabel="loss")
axes[1].set_yscale("log")
axes[1].legend(frameon=False, fontsize=7)
fig.tight_layout()

## 8. Best-Fit Reconstruction Quality

In [ ]:
best_replay = replay_fixation_mrnn_run(best_run_dir, device="cpu")
condition_order = tuple(best_replay["condition_order"])
region_order = tuple(best_replay["region_order"])

best_metrics = pd.concat(
    [
        reconstruction_accuracy(best_replay).assign(metric_space="region_pcs"),
        pc_reconstructed_firing_rate_accuracy(best_replay).assign(metric_space="backprojected_fr"),
    ],
    ignore_index=True,
)
fit_quality = (
    best_metrics.groupby(["metric_space", "region"], as_index=False)
    .agg(mean_mse=("mse", "mean"), mean_mae=("mae", "mean"), mean_r2=("r2", "mean"), mean_corr=("correlation", "mean"))
    .sort_values(["metric_space", "region"])
)
display(fit_quality)

## 9. Best-Fit Random-Region PC Reconstruction

In [ ]:
rng = np.random.default_rng()
pc_region = str(rng.choice(region_order))
n_pcs = min(6, targets.pcs_by_region[pc_region].shape[-1])

fig, axes = plt.subplots(n_pcs, len(condition_order), figsize=(3.2 * len(condition_order), 1.9 * n_pcs), sharex=True, squeeze=False)
for pc_idx in range(n_pcs):
    for cond_col, condition in enumerate(condition_order):
        ax = axes[pc_idx, cond_col]
        cond_idx = condition_order.index(condition)
        ax.plot(timeline, targets.pcs_by_region[pc_region][cond_idx, :, pc_idx], color="black", linewidth=2.0, label="target")
        yhat = best_replay["output_by_region"][pc_region].detach().cpu().numpy()[cond_idx, :, pc_idx]
        ax.plot(timeline, yhat, color="#2f6fbb", linewidth=1.2, label="model")
        ax.axvline(0.0, color="0.5", linewidth=0.7)
        if pc_idx == 0:
            ax.set_title(condition)
        if cond_col == 0:
            ax.set_ylabel(f"PC{pc_idx + 1}")
        if pc_idx == n_pcs - 1:
            ax.set_xlabel("time (s)")
axes[0, -1].legend(frameon=False, fontsize=7)
fig.suptitle(f"Best model region: {pc_region}", y=1.01)
fig.tight_layout()

## 10. Best-Fit Random-Region Backprojected Firing Rates

In [ ]:
fr_region = str(rng.choice(region_order))
target_fr_by_region = targets.pc_reconstructed_raw_by_region()
predicted_fr = backproject_replay_outputs_to_firing_rates(best_replay)[fr_region]
n_units = target_fr_by_region[fr_region].shape[-1]
unit_indices = np.sort(rng.choice(n_units, size=min(6, n_units), replace=False))

fig, axes = plt.subplots(len(unit_indices), len(condition_order), figsize=(3.2 * len(condition_order), 1.9 * len(unit_indices)), sharex=True, squeeze=False)
for row, unit_idx in enumerate(unit_indices):
    for cond_col, condition in enumerate(condition_order):
        ax = axes[row, cond_col]
        cond_idx = condition_order.index(condition)
        ax.plot(timeline, target_fr_by_region[fr_region][cond_idx, :, unit_idx], color="black", linewidth=2.0, label="target")
        ax.plot(timeline, predicted_fr[cond_idx, :, unit_idx], color="#2f6fbb", linewidth=1.2, label="model")
        ax.axvline(0.0, color="0.5", linewidth=0.7)
        if row == 0:
            ax.set_title(condition)
        if cond_col == 0:
            ax.set_ylabel(f"unit {unit_idx}")
        if row == len(unit_indices) - 1:
            ax.set_xlabel("time (s)")
axes[0, -1].legend(frameon=False, fontsize=7)
fig.suptitle(f"Best model region: {fr_region}", y=1.01)
fig.tight_layout()

## 11. Ensemble Current Projection Tables

This cell replays each trained model on CPU and computes current projections. It can take a few minutes because it loads all 100 checkpoints sequentially.

In [ ]:
condition_colors = {
    "face_interactive": "#b64198",
    "face_non_interactive": "#4c9a2a",
    "object": "#6f4e37",
}
region_colors = {
    "ofc": "#4c78a8",
    "bla": "#f58518",
    "dmpfc": "#54a24b",
    "accg": "#e45756",
}


def _region_slice(replay, region):
    start, stop = replay["model"].mrnn.get_region_indices(region)
    return slice(int(start), int(stop))


def _latent_arrays(replay, latent, region):
    sl = _region_slice(replay, region)
    conditions = tuple(replay["condition_order"])
    hidden = np.stack([latent[condition]["hidden_state"][:, sl].numpy() for condition in conditions], axis=0)
    drive = np.stack([latent[condition]["recurrent_drive"][:, sl].numpy() for condition in conditions], axis=0)
    return hidden, drive


def current_projection_table_for_replay(replay, init_idx, seed, active_condition):
    latent = extract_fixation_latent_dynamics(replay)
    current_vectors = extract_region_current_vectors(replay)
    conditions = tuple(replay["condition_order"])
    regions = tuple(replay["region_order"])
    active_idx = conditions.index(active_condition)
    rows = []
    for target_region in regions:
        hidden, drive = _latent_arrays(replay, latent, target_region)
        active_hidden = hidden[active_idx]
        reference_dirs = {
            condition: drive[conditions.index(condition)] - active_hidden
            for condition in conditions
        }
        for source_region in regions:
            current = current_vectors[(source_region, target_region)].numpy()[active_idx]
            for ref_condition, direction in reference_dirs.items():
                norm = np.linalg.norm(direction, axis=-1)
                unit_direction = np.divide(
                    direction,
                    np.maximum(norm[:, None], 1e-8),
                    out=np.zeros_like(direction),
                    where=norm[:, None] > 1e-8,
                )
                projection = np.sum(current * unit_direction, axis=-1)
                for time_idx, value in enumerate(projection):
                    rows.append(
                        {
                            "init_idx": int(init_idx),
                            "seed": int(seed),
                            "active_condition": active_condition,
                            "reference_condition": ref_condition,
                            "source_region": source_region,
                            "target_region": target_region,
                            "time_idx": int(time_idx),
                            "time_s": float(timeline[time_idx]),
                            "projection": float(value),
                        }
                    )
    return pd.DataFrame(rows)


def within_condition_projection_table_for_replay(replay, init_idx, seed):
    latent = extract_fixation_latent_dynamics(replay)
    current_vectors = extract_region_current_vectors(replay)
    conditions = tuple(replay["condition_order"])
    regions = tuple(replay["region_order"])
    rows = []
    for active_condition in conditions:
        active_idx = conditions.index(active_condition)
        for target_region in regions:
            hidden, drive = _latent_arrays(replay, latent, target_region)
            direction = drive[active_idx] - hidden[active_idx]
            norm = np.linalg.norm(direction, axis=-1)
            unit_direction = np.divide(
                direction,
                np.maximum(norm[:, None], 1e-8),
                out=np.zeros_like(direction),
                where=norm[:, None] > 1e-8,
            )
            source_projections = {}
            for source_region in regions:
                current = current_vectors[(source_region, target_region)].numpy()[active_idx]
                source_projections[source_region] = np.sum(current * unit_direction, axis=-1)
            denom = np.sum([np.abs(values) for values in source_projections.values()], axis=0)
            denom = np.where(denom > 1e-8, denom, 1.0)
            for source_region, projection in source_projections.items():
                relative_projection = projection / denom
                for time_idx, value in enumerate(projection):
                    rows.append(
                        {
                            "init_idx": int(init_idx),
                            "seed": int(seed),
                            "active_condition": active_condition,
                            "source_region": source_region,
                            "target_region": target_region,
                            "time_idx": int(time_idx),
                            "time_s": float(timeline[time_idx]),
                            "projection": float(value),
                            "relative_projection": float(relative_projection[time_idx]),
                        }
                    )
    return pd.DataFrame(rows)

current_projection_frames = []
within_projection_frames = []
for row in complete.itertuples(index=False):
    replay = replay_fixation_mrnn_run(row.run_dir, device="cpu")
    for active_condition in tuple(replay["condition_order"]):
        current_projection_frames.append(
            current_projection_table_for_replay(replay, row.init_idx, row.seed, active_condition)
        )
    within_projection_frames.append(within_condition_projection_table_for_replay(replay, row.init_idx, row.seed))

ensemble_current_projection_df = pd.concat(current_projection_frames, ignore_index=True)
ensemble_within_projection_df = pd.concat(within_projection_frames, ignore_index=True)
display(ensemble_current_projection_df.head())
display(ensemble_within_projection_df.head())

## 12. Mean Current Projection Grids by Fixation Type

In [ ]:
def _mean_sem(df, value_column, group_cols):
    out = df.groupby(group_cols, as_index=False)[value_column].agg(["mean", "sem"]).reset_index()
    return out.rename(columns={"mean": f"{value_column}_mean", "sem": f"{value_column}_sem"})


def plot_ensemble_current_projection_grid(active_condition, value_column="projection"):
    df = ensemble_current_projection_df[ensemble_current_projection_df["active_condition"] == active_condition]
    summary = _mean_sem(
        df,
        value_column,
        ["reference_condition", "source_region", "target_region", "time_idx", "time_s"],
    )
    fig, axes = plt.subplots(len(region_order), len(region_order), figsize=(12.5, 10.5), squeeze=False, sharex=True)
    for row, target_region in enumerate(region_order):
        for col, source_region in enumerate(region_order):
            ax = axes[row, col]
            subset = summary[(summary["target_region"] == target_region) & (summary["source_region"] == source_region)]
            ax.axhline(0.0, color="black", linewidth=0.6, alpha=0.5)
            ax.axvline(0.0, color="0.5", linewidth=0.6, alpha=0.45)
            for ref_condition in condition_order:
                trace = subset[subset["reference_condition"] == ref_condition].sort_values("time_idx")
                x = trace["time_s"].to_numpy(dtype=float)
                y = trace[f"{value_column}_mean"].to_numpy(dtype=float)
                sem = trace[f"{value_column}_sem"].fillna(0.0).to_numpy(dtype=float)
                color = condition_colors.get(ref_condition, "0.3")
                ax.plot(x, y, color=color, linewidth=1.0, label=ref_condition)
                ax.fill_between(x, y - sem, y + sem, color=color, alpha=0.16)
            if row == 0:
                ax.set_title(f"source {source_region}", fontsize=9)
            if col == 0:
                ax.set_ylabel(f"target {target_region}\\nmean projection")
            if row == len(region_order) - 1:
                ax.set_xlabel("time (s)")
    axes[0, -1].legend(frameon=False, fontsize=7, loc="upper left", bbox_to_anchor=(1.02, 1.0))
    fig.suptitle(f"Active fixation type: {active_condition}; mean +/- SEM across {n_initializations} seeds", y=1.01)
    fig.tight_layout()
    return fig, axes, summary

current_projection_summaries = {}
for active_condition in condition_order:
    fig, axes, summary = plot_ensemble_current_projection_grid(active_condition)
    current_projection_summaries[active_condition] = summary

## 13. Mean Within-Fixation Current Contribution Grid

In [ ]:
def plot_ensemble_within_condition_source_contribution_grid(value_column="relative_projection"):
    summary = _mean_sem(
        ensemble_within_projection_df,
        value_column,
        ["active_condition", "source_region", "target_region", "time_idx", "time_s"],
    )
    fig, axes = plt.subplots(
        len(condition_order),
        len(region_order),
        figsize=(13.0, 8.2),
        squeeze=False,
        sharex=True,
        sharey=value_column == "relative_projection",
    )
    for row, active_condition in enumerate(condition_order):
        for col, target_region in enumerate(region_order):
            ax = axes[row, col]
            subset = summary[(summary["active_condition"] == active_condition) & (summary["target_region"] == target_region)]
            ax.axhline(0.0, color="black", linewidth=0.6, alpha=0.5)
            ax.axvline(0.0, color="0.5", linewidth=0.6, alpha=0.45)
            for source_region in region_order:
                trace = subset[subset["source_region"] == source_region].sort_values("time_idx")
                x = trace["time_s"].to_numpy(dtype=float)
                y = trace[f"{value_column}_mean"].to_numpy(dtype=float)
                sem = trace[f"{value_column}_sem"].fillna(0.0).to_numpy(dtype=float)
                color = region_colors.get(source_region, "0.3")
                ax.plot(x, y, color=color, linewidth=1.0, label=source_region)
                ax.fill_between(x, y - sem, y + sem, color=color, alpha=0.16)
            if row == 0:
                ax.set_title(f"target {target_region}", fontsize=9)
            if col == 0:
                ax.set_ylabel(f"{active_condition}\\n{value_column}")
            if row == len(condition_order) - 1:
                ax.set_xlabel("time (s)")
            if value_column == "relative_projection":
                ax.set_ylim(-1.0, 1.0)
    axes[0, -1].legend(frameon=False, fontsize=7, loc="upper left", bbox_to_anchor=(1.02, 1.0))
    fig.suptitle(f"Within-fixation source alignment; mean +/- SEM across {n_initializations} seeds", y=1.02)
    fig.tight_layout()
    return fig, axes, summary

fig_within, axes_within, within_projection_summary = plot_ensemble_within_condition_source_contribution_grid()

## 14. Save Ensemble Summaries

In [ ]:
summary_dir = ensemble_dir / "ensemble_summaries"
summary_dir.mkdir(parents=True, exist_ok=True)
run_table.to_csv(summary_dir / "run_table.csv", index=False)
final_history.to_csv(summary_dir / "final_history.csv", index=False)
training_health.to_csv(summary_dir / "training_health.csv", index=False)
ensemble_current_projection_df.to_pickle(summary_dir / "current_projection_by_seed.pkl")
ensemble_within_projection_df.to_pickle(summary_dir / "within_current_projection_by_seed.pkl")
within_projection_summary.to_csv(summary_dir / "within_current_projection_mean_sem.csv", index=False)
for condition, summary in current_projection_summaries.items():
    summary.to_csv(summary_dir / f"current_projection_mean_sem_{condition}.csv", index=False)
print(summary_dir)